# Experiment: Product Vectorisation + 3D UMAP

This notebook vectorises products using `TITLE`, `BULLET_POINTS`, and `DESCRIPTION`, stores vectors locally, and plots a 3D UMAP projection.

It is designed for large CSVs using chunked embedding.

In [24]:
%pip install -q --upgrade pandas sentence-transformers umap-learn scikit-learn plotly pyarrow tqdm nbformat ipykernel

Note: you may need to restart the kernel to use updated packages.


In [25]:
from pathlib import Path
import csv
import json
import numpy as np
import pandas as pd
import plotly.express as px
import umap.umap_ as umap
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

In [26]:
# Configuration
DATASET_GLOB = "data/raw/amazon-product-data/**/*.csv"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Optional cap before sampling. Set None to use full dataset.
MAX_ROWS = None

# Fraction of rows to embed from the selected population (e.g., 0.01 = 1%).
SAMPLE_FRAC = 0.01

CHUNK_SIZE = 10000
BATCH_SIZE = 128
UMAP_SAMPLE_SIZE = 50000
PLOT_MAX_POINTS = 5000
RANDOM_STATE = 42


In [27]:
# Locate dataset and project root
search_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
csv_paths = []
project_root = None

for root in search_roots:
    candidate = sorted(root.glob(DATASET_GLOB))
    if candidate:
        csv_paths = candidate
        project_root = root.resolve()
        break

if not csv_paths:
    searched = [str(r / DATASET_GLOB) for r in search_roots]
    msg = "No CSV found. Searched:\n" + "\n".join(searched)
    raise FileNotFoundError(msg)

dataset_path = csv_paths[0].resolve()
ARTIFACT_DIR = project_root / "experiment_vectorisation" / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")
print(f"Using dataset: {dataset_path}")
print(f"Artifact dir: {ARTIFACT_DIR}")


Project root: /Users/lohzh/Desktop/cs3263-repo
Using dataset: /Users/lohzh/Desktop/cs3263-repo/data/raw/amazon-product-data/dataset/train.csv
Artifact dir: /Users/lohzh/Desktop/cs3263-repo/experiment_vectorisation/artifacts


In [28]:
# Discover columns
schema_df = pd.read_csv(dataset_path, nrows=0)
all_columns = schema_df.columns.tolist()
print(all_columns)

def resolve_column(columns, candidates):
    lookup = {c.lower(): c for c in columns}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]
    return None

col_id = resolve_column(all_columns, ["PRODUCT_ID", "product_id", "id"])
col_title = resolve_column(all_columns, ["TITLE", "title", "product_title"])
col_bullets = resolve_column(all_columns, ["BULLET_POINTS", "bullet_points", "bullets"])
col_desc = resolve_column(all_columns, ["DESCRIPTION", "description", "product_description"])
col_type = resolve_column(all_columns, ["PRODUCT_TYPE_ID", "product_type_id", "category", "label"])

if col_title is None:
    raise ValueError("Could not find a title column.")

text_usecols = [c for c in [col_id, col_title, col_bullets, col_desc, col_type] if c is not None]
print("Using columns:", text_usecols)

['PRODUCT_ID', 'TITLE', 'BULLET_POINTS', 'DESCRIPTION', 'PRODUCT_TYPE_ID', 'PRODUCT_LENGTH']
Using columns: ['PRODUCT_ID', 'TITLE', 'BULLET_POINTS', 'DESCRIPTION', 'PRODUCT_TYPE_ID']


In [29]:
def normalize_text(x):
    if pd.isna(x):
        return ""
    t = str(x).strip()
    if t.startswith("[") and t.endswith("]"):
        t = t[1:-1]
    return " ".join(t.replace("\n", " ").split())


def build_document_text(frame: pd.DataFrame) -> pd.Series:
    title = frame[col_title].map(normalize_text)
    bullets = frame[col_bullets].map(normalize_text) if col_bullets else ""
    desc = frame[col_desc].map(normalize_text) if col_desc else ""
    return "Title: " + title + " | Bullet Points: " + bullets + " | Description: " + desc

In [30]:
# Count rows and determine exact random sample size.
with dataset_path.open("r", encoding="utf-8", errors="ignore") as f:
    total_rows_in_csv = sum(1 for _ in f) - 1

population_rows = total_rows_in_csv if MAX_ROWS is None else min(MAX_ROWS, total_rows_in_csv)

if SAMPLE_FRAC is None:
    target_rows = population_rows
    selected_row_positions = None
else:
    if not (0 < SAMPLE_FRAC <= 1):
        raise ValueError("SAMPLE_FRAC must be in (0, 1].")
    target_rows = max(1, int(population_rows * SAMPLE_FRAC))
    rng = np.random.default_rng(RANDOM_STATE)
    selected_row_positions = np.sort(rng.choice(population_rows, size=target_rows, replace=False))

print(f"Total rows in CSV: {total_rows_in_csv:,}")
print(f"Population rows considered: {population_rows:,}")
print(f"Rows to embed: {target_rows:,}")


Total rows in CSV: 2,249,698
Population rows considered: 2,249,698
Rows to embed: 22,496


In [31]:
# Embed in chunks and persist artifacts
model = SentenceTransformer(MODEL_NAME)
embedding_dim = model.get_sentence_embedding_dimension()

emb_path = ARTIFACT_DIR / "product_embeddings.float32.memmap"
meta_path = ARTIFACT_DIR / "product_metadata.csv"
sample_emb_path = ARTIFACT_DIR / "umap_sample_embeddings.npy"
sample_meta_path = ARTIFACT_DIR / "umap_sample_metadata.parquet"
run_info_path = ARTIFACT_DIR / "run_info.json"

if meta_path.exists():
    meta_path.unlink()

embeddings_mm = np.memmap(emb_path, mode="w+", dtype="float32", shape=(target_rows, embedding_dim))

meta_cols = [c for c in [col_id, col_title, col_bullets, col_desc, col_type] if c is not None]

processed = 0
sample_emb_list = []
sample_meta_list = []
remaining_for_sample = UMAP_SAMPLE_SIZE

source_row_cursor = 0
selected_ptr = 0

reader = pd.read_csv(dataset_path, usecols=text_usecols, chunksize=CHUNK_SIZE)
for chunk in tqdm(reader, desc="Embedding chunks"):
    if source_row_cursor >= population_rows or processed >= target_rows:
        break

    raw_n = len(chunk)
    if source_row_cursor + raw_n > population_rows:
        chunk = chunk.iloc[: population_rows - source_row_cursor].copy()
        raw_n = len(chunk)

    if selected_row_positions is not None:
        start_ptr = selected_ptr
        upper = source_row_cursor + raw_n
        while selected_ptr < target_rows and selected_row_positions[selected_ptr] < upper:
            selected_ptr += 1

        if selected_ptr == start_ptr:
            source_row_cursor += raw_n
            continue

        rel_idx = selected_row_positions[start_ptr:selected_ptr] - source_row_cursor
        chunk = chunk.iloc[rel_idx].copy()

    source_row_cursor += raw_n

    chunk["document_text"] = build_document_text(chunk)
    chunk = chunk[chunk["document_text"].str.strip().str.len() > 0].copy()
    if len(chunk) == 0:
        continue

    vectors = model.encode(
        chunk["document_text"].tolist(),
        batch_size=BATCH_SIZE,
        convert_to_numpy=True,
        show_progress_bar=False,
        normalize_embeddings=True,
    ).astype("float32")

    n = len(chunk)
    embeddings_mm[processed : processed + n] = vectors

    chunk_meta = chunk[meta_cols].copy()
    chunk_meta.insert(0, "row_index", np.arange(processed, processed + n, dtype=np.int64))
    chunk_meta["document_text"] = chunk["document_text"].values

    chunk_meta.to_csv(meta_path, mode="a", index=False, header=not meta_path.exists())

    if remaining_for_sample > 0:
        take = min(remaining_for_sample, max(1, int(np.ceil(n * UMAP_SAMPLE_SIZE / max(target_rows, 1)))))
        sampled = chunk_meta.sample(n=min(take, len(chunk_meta)), random_state=RANDOM_STATE)
        sample_meta_list.append(sampled)
        sample_emb_list.append(vectors[sampled["row_index"].to_numpy() - processed])
        remaining_for_sample -= len(sampled)

    processed += n

if processed == 0:
    raise RuntimeError("No rows were embedded. Check source columns and text preprocessing.")

embeddings_mm.flush()

if processed < target_rows:
    print(f"Processed {processed:,} rows (less than target {target_rows:,} after cleaning).")

sample_emb = np.vstack(sample_emb_list) if sample_emb_list else embeddings_mm[: min(processed, UMAP_SAMPLE_SIZE)].copy()
sample_meta_df = pd.concat(sample_meta_list, ignore_index=True) if sample_meta_list else pd.read_csv(meta_path, nrows=len(sample_emb))

np.save(sample_emb_path, sample_emb)
sample_meta_df.to_parquet(sample_meta_path, index=False)

run_info = {
    "dataset_path": str(dataset_path),
    "rows_in_csv": int(total_rows_in_csv),
    "population_rows": int(population_rows),
    "sample_frac": SAMPLE_FRAC,
    "rows_embedded": int(processed),
    "embedding_dim": int(embedding_dim),
    "model_name": MODEL_NAME,
    "embedding_file": str(emb_path),
    "metadata_file": str(meta_path),
    "umap_sample_embeddings_file": str(sample_emb_path),
    "umap_sample_metadata_file": str(sample_meta_path),
}
run_info_path.write_text(json.dumps(run_info, indent=2))

print(f"Rows embedded: {processed:,}")
print(f"Saved embeddings: {emb_path}")
print(f"Saved metadata: {meta_path}")
print(f"Saved run info: {run_info_path}")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7458.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Embedding chunks: 225it [01:31,  2.47it/s]

Rows embedded: 22,496
Saved embeddings: /Users/lohzh/Desktop/cs3263-repo/experiment_vectorisation/artifacts/product_embeddings.float32.memmap
Saved metadata: /Users/lohzh/Desktop/cs3263-repo/experiment_vectorisation/artifacts/product_metadata.csv
Saved run info: /Users/lohzh/Desktop/cs3263-repo/experiment_vectorisation/artifacts/run_info.json


In [32]:
# 3D UMAP on sampled embeddings
umap_model = umap.UMAP(
    n_components=3,
    metric="cosine",
    n_neighbors=15,
    min_dist=0.05,
    random_state=RANDOM_STATE,
)

umap_3d = umap_model.fit_transform(sample_emb)

umap_df = sample_meta_df.copy().reset_index(drop=True)
umap_df["umap_x"] = umap_3d[:, 0]
umap_df["umap_y"] = umap_3d[:, 1]
umap_df["umap_z"] = umap_3d[:, 2]

umap_path = ARTIFACT_DIR / "umap_3d_sample.parquet"
umap_df.to_parquet(umap_path, index=False)
print(f"Saved UMAP coordinates: {umap_path}")

/Users/lohzh/Desktop/cs3263-repo/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved UMAP coordinates: /Users/lohzh/Desktop/cs3263-repo/experiment_vectorisation/artifacts/umap_3d_sample.parquet


In [33]:
# Plot interactive 3D UMAP
plot_df = umap_df.sample(n=min(PLOT_MAX_POINTS, len(umap_df)), random_state=RANDOM_STATE)

color_col = col_type if col_type in plot_df.columns else None
if color_col is not None:
    plot_df[color_col] = plot_df[color_col].astype(str)

hover_cols = [c for c in [col_id, col_title, col_type] if c in plot_df.columns]

fig = px.scatter_3d(
    plot_df,
    x="umap_x",
    y="umap_y",
    z="umap_z",
    color=color_col,
    hover_data=hover_cols,
    opacity=0.75,
    title=f"3D UMAP of Product Embeddings ({len(plot_df):,} sampled points)",
)
fig.update_traces(marker=dict(size=3))

try:
    fig.show()
except ValueError as e:
    if "nbformat" in str(e):
        import plotly.io as pio
        pio.renderers.default = "browser"
        print("Falling back to browser renderer. Install nbformat/ipykernel and rerun install cell for inline rendering.")
        fig.show()
    else:
        raise

Falling back to browser renderer. Install nbformat/ipykernel and rerun install cell for inline rendering.
